In [2]:
# streamlit_app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

st.set_page_config(page_title="Journal Explorer", layout="wide")

st.title("Journal Explorer")
st.write("Explore journals by year, citations, and semantic similarity.")

@st.cache_data
def load_data():
    journals_display = pd.read_csv("journals_display.csv", index_col=0)
    journals_model_final = pd.read_csv("../1_sources/journals_model_final.csv", index_col=0)
    return journals_display, journals_model_final

@st.cache_resource
def load_models():
    tfidf = joblib.load("tfidf_vectorizer.pkl")
    pca = joblib.load("pca_model.pkl")
    scaler = joblib.load("scaler.pkl")
    mlb = joblib.load("mlb.pkl")
    return tfidf, pca, scaler, mlb

journals_display, journals_model_final = load_data()
tfidf, pca, scaler, mlb = load_models()

df = journals_display.copy()

for col in ["year", "citations"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "year" not in df.columns:
    st.error("Missing year column in journals_display.csv")
    st.stop()

years = sorted(df["year"].dropna().astype(int).unique().tolist())
if not years:
    st.error("No valid years found.")
    st.stop()

st.sidebar.header("Filters")
year_min, year_max = int(min(years)), int(max(years))
selected_years = st.sidebar.slider(
    "Select year range",
    min_value=year_min,
    max_value=year_max,
    value=(year_min, year_max),
    step=1
)

search_text = st.sidebar.text_input("Search title / journal / abstract")
min_citations = st.sidebar.slider("Minimum citations", 0, int(df["citations"].fillna(0).max() if "citations" in df.columns else 0), 0)

filtered = df.copy()
filtered = filtered[filtered["year"].between(selected_years[0], selected_years[1], inclusive="both")]

if "citations" in filtered.columns:
    filtered = filtered[filtered["citations"].fillna(0) >= min_citations]

if search_text:
    mask = pd.Series(False, index=filtered.index)
    for col in filtered.columns:
        if filtered[col].dtype == "object":
            mask = mask | filtered[col].astype(str).str.contains(search_text, case=False, na=False)
    filtered = filtered[mask]

tab1, tab2, tab3, tab4 = st.tabs(["Overview", "Yearly trend", "Citation analysis", "Results"])

with tab1:
    c1, c2, c3 = st.columns(3)
    c1.metric("Papers shown", len(filtered))
    c2.metric("Years selected", f"{selected_years[0]} - {selected_years[1]}")
    if "citations" in filtered.columns:
        c3.metric("Median citations", f"{filtered['citations'].median():.0f}" if len(filtered) else "0")

    st.subheader("Year selection")
    yearly = df.groupby("year").size().reset_index(name="count").dropna()
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(yearly["year"], yearly["count"], marker="o", linewidth=2, color="#7F77DD")
    ax.fill_between(yearly["year"], yearly["count"], alpha=0.15, color="#7F77DD")
    ax.axvspan(selected_years[0], selected_years[1], color="orange", alpha=0.15)
    ax.set_title("Publications per year")
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of articles")
    st.pyplot(fig)

with tab2:
    yearly = df.groupby("year").size().reset_index(name="count").dropna()
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(yearly["year"], yearly["count"], color="#7F77DD")
    ax.set_title("Publications per year")
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of articles")
    st.pyplot(fig)

    st.subheader("Year distribution")
    st.dataframe(yearly, use_container_width=True)

with tab3:
    if "citations" in df.columns:
        clean_citations = df["citations"].dropna()
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Mean citations", f"{clean_citations.mean():.1f}" if len(clean_citations) else "0")
        c2.metric("Median citations", f"{clean_citations.median():.1f}" if len(clean_citations) else "0")
        c3.metric("Zero citations", int((clean_citations == 0).sum()))
        c4.metric("> 100 citations", int((clean_citations > 100).sum()))

        fig, ax = plt.subplots(figsize=(10, 4))
        clean_citations.clip(upper=200).hist(bins=50, color="#1D9E75", edgecolor="white", ax=ax)
        ax.set_title("Citation distribution clipped at 200")
        ax.set_xlabel("Citations")
        ax.set_ylabel("Count")
        st.pyplot(fig)

        fig2, ax2 = plt.subplots(figsize=(10, 4))
        ax2.scatter(df["year"], df["citations"], alpha=0.35, s=20, color="#4444aa")
        ax2.set_title("Citations by year")
        ax2.set_xlabel("Year")
        ax2.set_ylabel("Citations")
        st.pyplot(fig2)
    else:
        st.info("No citations column found.")

with tab4:
    st.subheader("Filtered journals")
    st.dataframe(filtered, use_container_width=True)

    csv = filtered.to_csv(index=False).encode("utf-8")
    st.download_button("Download filtered data", csv, "filtered_journals.csv", "text/csv")

st.divider()
st.subheader("Optional: model-ready query transform")

query = st.text_input("Enter a title or abstract to transform")
if query:
    q_tfidf = tfidf.transform([query])
    q_pca = pca.transform(q_tfidf.toarray())
    st.write("TF-IDF shape:", q_tfidf.shape)
    st.write("PCA shape:", q_pca.shape)



2026-06-11 15:29:22.503 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 15:29:22.504 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 15:29:22.594 
  command:

    streamlit run /opt/anaconda3/lib/python3.13/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-06-11 15:29:22.594 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 15:29:22.594 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 15:29:22.595 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 15:29:22.595 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when run

FileNotFoundError: [Errno 2] No such file or directory: 'journals_display.csv'

NameError: name 'df' is not defined